# Shared walls in one building

Use RC 0.6.0, Core 0.6.0 and Data 0.6.0 with the platform and notebook extras.
All geometry is illustrative and in the same metric frame. A supplied neighbouring
wall extrusion creates an explicit adiabatic contact assumption; no neighbouring
temperature is modelled. Core keeps windows on outdoor portions and retains the
installed opaque construction. RC uses the packaged Goteborg EPW, with explicit
ground coupling and supply-control omissions. Outputs stay in memory.


In [ ]:
from importlib.resources import files
import numpy as np
from epsm_core import Building, Footprint, GeometrySpec, PartyWallNeighbor, Weather
from epsm_rc import PreparationCache, RCSettings, simulate_building

geometry = GeometrySpec.from_footprint(
    Footprint.rectangle(10, 8), storeys=2, storey_height_m=3)
neighbor = PartyWallNeighbor(
    identifier="next", footprint=Footprint(
        exterior=((2, -5), (7, -5), (7, 0), (2, 0))),
    bottom_m=1, top_m=4)
contact_geometry = geometry.with_party_wall_contacts(
    (neighbor,), tolerance_m=0,
    assumption_source="Illustrative exact wall contact; adiabatic approximation")
building = Building(preset="illustrative-reference", geometry=contact_geometry)
assert building.readiness(engine="epsm_rc").ready
print(contact_geometry.party_walls)


The common wall patch is 5 m wide and 3 m high. It crosses a storey boundary,
but its physical area remains 15 m². Window elements are outdoor. The rest of the
facade remains exposed. Neighbour eave height and the wall outline must be supplied;
proximity alone does not establish wall contact.


In [ ]:
shared_area = sum(element.area_m2 for element in building.elements
                  if element.boundary == "Adiabatic")
assert np.isclose(shared_area, 15)
assert all(element.boundary == "Outdoors" for element in building.elements
           if element.role.value == "WINDOW")
weather = Weather.from_epw(files("epsm_rc").joinpath("data/epw/goteborg.epw"))
settings = RCSettings.from_policy(
    "epsm_rc_5r1c_v2", airflow_policy="sensible_only_unlimited_supply",
    ground_policy="conductance_multiplier")
cache = PreparationCache()
first = simulate_building(building, weather=weather, settings=settings)
second = simulate_building(building, weather=weather, settings=settings,
                           preparation_cache=cache)
assert first.run_id == second.run_id
for name, values in first.hourly.items():
    np.testing.assert_array_equal(values, second.hourly[name])
assert first.diagnostics["boundary_reduction"]["adiabatic_opaque_area_m2"] == 15
print(first.diagnostics["boundary_reduction"])
print(first.scalars["space_heating_demand_kwh"])


Shared and internal opaque areas retain installed material and the existing
room-side effective-depth storage approximation. No heat, solar absorption or sky
exchange is assigned through an adiabatic patch. The aggregated RC domain does not
resolve separate room temperatures. These assertions verify the software handoff;
they do not validate the assumed construction or energy against measured buildings.
